# Data Cleaning: Extended Movie Dataset


In [1]:
# Import packages used for data loading, parsing, and feature construction
import ast
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
# Define file paths for raw inputs and processed outputs
RAW_DIR = Path("../data/raw")
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MOVIES_METADATA_PATH = RAW_DIR / "movies_metadata.csv"
CREDITS_PATH = RAW_DIR / "credits.csv"
RATINGS_PATH = RAW_DIR / "ratings.csv"
LINKS_PATH = RAW_DIR / "links.csv"
BECHDEL_PATH = RAW_DIR / "Bechdel_IMDB_Merge0524.csv"
OSCAR_PATH = RAW_DIR / "the_oscar_award.csv"

In [3]:
# Define helper functions for nested fields, text cleaning, and title matching

def safe_literal_eval(value):
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []


# Extract name fields from parsed list records.
def get_names(value, key="name"):
    return [item.get(key) for item in safe_literal_eval(value) if isinstance(item, dict) and item.get(key)]


def get_first_name(value):
    names = get_names(value)
    return names[0] if names else np.nan


# Normalize titles for cross-source matching.
def normalize_title(value):
    if pd.isna(value):
        return ""
    value = str(value).lower().strip()
    value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("ascii")
    value = re.sub(r"^(the|a|an)\s+", "", value)
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


# Normalize person names for text comparisons.
def normalize_name(value):
    value = str(value).lower().strip()
    value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("ascii")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


# Calculate female share among records with known binary gender labels.
def gender_ratio(records, top_n=None):
    people = records[:top_n] if top_n is not None else records
    known = [person for person in people if person.get("gender") in [1, 2]]
    if not known:
        return np.nan
    return sum(person.get("gender") == 1 for person in known) / len(known)


# Flag whether a specified crew job is held by a woman.
def has_female_job(crew_records, jobs):
    jobs = set(jobs)
    return int(any(person.get("job") in jobs and person.get("gender") == 1 for person in crew_records))


# Calculate female share across crew records with known binary gender labels.
def crew_gender_ratio(crew_records):
    known = [person for person in crew_records if person.get("gender") in [1, 2]]
    if not known:
        return np.nan
    return sum(person.get("gender") == 1 for person in known) / len(known)

In [4]:
# Load movie metadata and standardize base movie fields
movies = pd.read_csv(MOVIES_METADATA_PATH, low_memory=False)

movies["tmdb_id"] = pd.to_numeric(movies["id"], errors="coerce")
movies = movies.dropna(subset=["tmdb_id"]).copy()
movies["tmdb_id"] = movies["tmdb_id"].astype(int)

movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
movies["year"] = movies["release_date"].dt.year
movies["release_decade"] = (movies["year"] // 10 * 10).astype("Int64")

for col in ["budget", "revenue", "popularity", "runtime", "vote_average", "vote_count"]:
    movies[col] = pd.to_numeric(movies[col], errors="coerce")

movies["genre1"] = movies["genres"].apply(lambda x: get_names(x)[0] if len(get_names(x)) > 0 else np.nan)
movies["genre2"] = movies["genres"].apply(lambda x: get_names(x)[1] if len(get_names(x)) > 1 else np.nan)
movies["genre3"] = movies["genres"].apply(lambda x: get_names(x)[2] if len(get_names(x)) > 2 else np.nan)
movies["production_country"] = movies["production_countries"].apply(get_first_name)
movies["main_production_company"] = movies["production_companies"].apply(get_first_name)
movies["title_clean"] = movies["title"].apply(normalize_title)

movie_cols = [
    "tmdb_id", "imdb_id", "title", "title_clean", "year", "release_decade", "release_date",
    "budget", "revenue", "popularity", "runtime", "vote_average", "vote_count",
    "original_language", "genre1", "genre2", "genre3", "production_country", "main_production_company",
]
movies_clean = movies[movie_cols].drop_duplicates(subset=["tmdb_id"]).copy()

print("Movies metadata:", movies_clean.shape)
movies_clean.head()

Movies metadata: (45433, 19)


,tmdb_id,imdb_id,title,title_clean,year,release_decade,release_date,budget,revenue,popularity,runtime,vote_average,vote_count,original_language,genre1,genre2,genre3,production_country,main_production_company
0,862,tt0114709,Toy Story,toy story,1995.0,1990,1995-10-30,30000000,373554033.0,21.946943,81.0,7.7,5415.0,en,Animation,Comedy,Family,United States of America,Pixar Animation Studios
1,8844,tt0113497,Jumanji,jumanji,1995.0,1990,1995-12-15,65000000,262797249.0,17.015539,104.0,6.9,2413.0,en,Adventure,Fantasy,Family,United States of America,TriStar Pictures
2,15602,tt0113228,Grumpier Old Men,grumpier old men,1995.0,1990,1995-12-22,0,0.0,11.712900,101.0,6.5,92.0,en,Romance,Comedy,NaN,United States of America,Warner Bros.
3,31357,tt0114885,Waiting to Exhale,waiting to exhale,1995.0,1990,1995-12-22,16000000,81452156.0,3.859495,127.0,6.1,34.0,en,Comedy,Drama,Romance,United States of America,Twentieth Century Fox Film Corporation
4,11862,tt0113041,Father of the Bride Part II,father of the bride part ii,1995.0,1990,1995-02-10,0,76578911.0,8.387519,106.0,5.7,173.0,en,Comedy,NaN,NaN,United States of America,Sandollar Productions


In [5]:
# Load credits and calculate cast and crew gender representation variables
credits = pd.read_csv(CREDITS_PATH)
credits["tmdb_id"] = pd.to_numeric(credits["id"], errors="coerce")
credits = credits.dropna(subset=["tmdb_id"]).copy()
credits["tmdb_id"] = credits["tmdb_id"].astype(int)

# Parse nested cast and crew records before computing representation measures.
credits["cast_records"] = credits["cast"].apply(safe_literal_eval)
credits["crew_records"] = credits["crew"].apply(safe_literal_eval)

# Calculate female cast share at different billing depths.
credits["female_ratio_all_cast"] = credits["cast_records"].apply(gender_ratio)
credits["female_ratio_top3"] = credits["cast_records"].apply(lambda x: gender_ratio(x, top_n=3))
credits["female_ratio_top5"] = credits["cast_records"].apply(lambda x: gender_ratio(x, top_n=5))
credits["female_ratio_top10"] = credits["cast_records"].apply(lambda x: gender_ratio(x, top_n=10))
credits["female_lead"] = credits["cast_records"].apply(lambda x: int(len(x) > 0 and x[0].get("gender") == 1))
credits["female_second_billed"] = credits["cast_records"].apply(lambda x: int(len(x) > 1 and x[1].get("gender") == 1))

# Create binary indicators for women in selected crew roles.
credits["female_director"] = credits["crew_records"].apply(lambda x: has_female_job(x, ["Director"]))
credits["female_writer"] = credits["crew_records"].apply(lambda x: has_female_job(x, ["Writer", "Screenplay", "Story", "Original Story", "Novel"]))
credits["female_producer"] = credits["crew_records"].apply(lambda x: has_female_job(x, ["Producer"]))
credits["female_executive_producer"] = credits["crew_records"].apply(lambda x: has_female_job(x, ["Executive Producer"]))
credits["female_editor"] = credits["crew_records"].apply(lambda x: has_female_job(x, ["Editor"]))
credits["female_cinematographer"] = credits["crew_records"].apply(lambda x: has_female_job(x, ["Director of Photography", "Cinematography"]))
credits["female_crew_ratio"] = credits["crew_records"].apply(crew_gender_ratio)

representation_flags = [
    "female_director", "female_writer", "female_producer", "female_executive_producer",
    "female_editor", "female_cinematographer",
]
credits["female_creative_leadership_score"] = credits[representation_flags].mean(axis=1)

credits_clean = credits[[
    "tmdb_id", "female_ratio_all_cast", "female_ratio_top3", "female_ratio_top5", "female_ratio_top10",
    "female_lead", "female_second_billed", "female_director", "female_writer", "female_producer",
    "female_executive_producer", "female_editor", "female_cinematographer", "female_crew_ratio",
    "female_creative_leadership_score",
]].drop_duplicates(subset=["tmdb_id"]).copy()

print("Credits features:", credits_clean.shape)
credits_clean.head()

Credits features: (45432, 15)


,tmdb_id,female_ratio_all_cast,female_ratio_top3,female_ratio_top5,female_ratio_top10,female_lead,female_second_billed,female_director,female_writer,female_producer,female_executive_producer,female_editor,female_cinematographer,female_crew_ratio,female_creative_leadership_score
0,862,0.250000,0.000000,0.0,0.222222,0,0,0,0,1,0,0,0,0.121212,0.166667
1,8844,0.400000,0.333333,0.5,0.625000,0,0,0,0,0,0,0,0,0.000000,0.000000
2,15602,0.428571,0.333333,0.6,0.428571,0,0,0,0,0,0,0,0,0.000000,0.000000
3,31357,0.400000,1.000000,0.8,0.400000,1,1,0,0,1,0,0,0,0.250000,0.166667
4,11862,0.454545,0.333333,0.4,0.444444,0,1,0,1,1,0,0,0,0.285714,0.333333


In [6]:
# Aggregate MovieLens ratings and join them to movie records through TMDB IDs
rating_chunks = []
for chunk in pd.read_csv(RATINGS_PATH, usecols=["movieId", "rating"], chunksize=1_000_000):
    rating_chunks.append(chunk.groupby("movieId")["rating"].agg(["count", "sum"]))

ratings_agg = pd.concat(rating_chunks).groupby(level=0).sum().reset_index()
ratings_agg["movielens_avg_rating"] = ratings_agg["sum"] / ratings_agg["count"]
ratings_agg = ratings_agg.rename(columns={"count": "movielens_rating_count"})[[
    "movieId", "movielens_rating_count", "movielens_avg_rating"
]]

links = pd.read_csv(LINKS_PATH)
links["tmdb_id"] = pd.to_numeric(links["tmdbId"], errors="coerce")
links = links.dropna(subset=["tmdb_id"]).copy()
links["tmdb_id"] = links["tmdb_id"].astype(int)
links["imdb_id_from_links"] = "tt" + links["imdbId"].astype(str).str.zfill(7)

ratings_clean = links[["movieId", "tmdb_id", "imdb_id_from_links"]].merge(ratings_agg, on="movieId", how="left")
ratings_clean = ratings_clean.drop_duplicates(subset=["tmdb_id"])

print("MovieLens ratings linked:", ratings_clean.shape)
ratings_clean.head()

MovieLens ratings linked: (45594, 5)


,movieId,tmdb_id,imdb_id_from_links,movielens_rating_count,movielens_avg_rating
0,1,862,tt0114709,66008.0,3.888157
1,2,8844,tt0113497,26060.0,3.236953
2,3,15602,tt0113228,15497.0,3.175550
3,4,31357,tt0114885,2981.0,2.875713
4,5,11862,tt0113041,15258.0,3.079565


In [7]:
# Prepare Bechdel variables and align them by IMDb identifier
bechdel = pd.read_csv(BECHDEL_PATH)
bechdel["imdb_id"] = "tt" + bechdel["imdbid"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(7)
bechdel["bechdel_pass"] = (bechdel["bechdelRating"] == 3).astype(int)

bechdel_clean = bechdel[[
    "imdb_id", "bechdelRating", "bechdel_pass", "imdbAverageRating", "numVotes", "runtimeMinutes"
]].drop_duplicates(subset=["imdb_id"]).copy()

print("Bechdel features:", bechdel_clean.shape)
bechdel_clean.head()

Bechdel features: (9710, 6)


,imdb_id,bechdelRating,bechdel_pass,imdbAverageRating,numVotes,runtimeMinutes
0,tt0000009,0,0,5.4,212.0,45
1,tt0000574,1,0,6.0,903.0,70
2,tt0002101,2,0,5.1,622.0,100
3,tt0003973,2,0,5.8,300.0,63
4,tt0004972,2,0,6.1,26403.0,195


In [8]:
# Prepare Oscar nomination and win variables at the movie level
oscar = pd.read_csv(OSCAR_PATH)
oscar_film = oscar.dropna(subset=["film"]).copy()
oscar_film["title_clean"] = oscar_film["film"].apply(normalize_title)
oscar_film["year"] = oscar_film["year_film"]
oscar_film["oscar_nomination"] = 1
oscar_film["oscar_win"] = oscar_film["winner"].astype(int)
oscar_film["oscar_directing_nomination"] = oscar_film["canon_category"].eq("DIRECTING").astype(int)
oscar_film["oscar_directing_win"] = (oscar_film["canon_category"].eq("DIRECTING") & oscar_film["winner"]).astype(int)
oscar_film["oscar_best_picture_nomination"] = oscar_film["canon_category"].eq("BEST PICTURE").astype(int)
oscar_film["oscar_best_picture_win"] = (oscar_film["canon_category"].eq("BEST PICTURE") & oscar_film["winner"]).astype(int)

oscar_clean = (
    oscar_film.groupby(["title_clean", "year"], as_index=False)
    .agg(
        oscar_nomination_count=("oscar_nomination", "sum"),
        oscar_win_count=("oscar_win", "sum"),
        oscar_directing_nomination=("oscar_directing_nomination", "max"),
        oscar_directing_win=("oscar_directing_win", "max"),
        oscar_best_picture_nomination=("oscar_best_picture_nomination", "max"),
        oscar_best_picture_win=("oscar_best_picture_win", "max"),
    )
)

print("Oscar film features:", oscar_clean.shape)
oscar_clean.head()

Oscar film features: (5231, 8)


,title_clean,year,oscar_nomination_count,oscar_win_count,oscar_directing_nomination,oscar_directing_win,oscar_best_picture_nomination,oscar_best_picture_win
0,1 000 a minute,1935,1,0,0,0,0,0
1,10,1979,2,0,0,0,0,0
2,100 year old man who climbed out the window an...,2015,1,0,0,0,0,0
3,102 dalmatians,2000,1,0,0,0,0,0
4,12,2007,1,0,0,0,0,0


In [9]:
# Merge metadata, representation, rating, Bechdel, and Oscar feature tables
movie_dataset = movies_clean.merge(credits_clean, on="tmdb_id", how="left")
movie_dataset = movie_dataset.merge(
    ratings_clean[["tmdb_id", "movieId", "movielens_rating_count", "movielens_avg_rating"]],
    on="tmdb_id",
    how="left",
)
movie_dataset = movie_dataset.merge(bechdel_clean, on="imdb_id", how="left")
movie_dataset = movie_dataset.merge(oscar_clean, on=["title_clean", "year"], how="left")

oscar_cols = [
    "oscar_nomination_count", "oscar_win_count", "oscar_directing_nomination", "oscar_directing_win",
    "oscar_best_picture_nomination", "oscar_best_picture_win",
]
movie_dataset[oscar_cols] = movie_dataset[oscar_cols].fillna(0).astype(int)

# Preserve raw financial columns before cleaning budget and revenue values.
# Treat non-positive budget and revenue values as missing financial data.
movie_dataset["budget_raw"] = movie_dataset["budget"]
movie_dataset["revenue_raw"] = movie_dataset["revenue"]
movie_dataset["budget"] = movie_dataset["budget"].where(movie_dataset["budget"] > 0, np.nan)
movie_dataset["revenue"] = movie_dataset["revenue"].where(movie_dataset["revenue"] > 0, np.nan)

movie_dataset["has_budget_data"] = movie_dataset["budget"].notna().astype(int)
movie_dataset["has_revenue_data"] = movie_dataset["revenue"].notna().astype(int)
movie_dataset["has_financial_data"] = (movie_dataset["budget"].notna() & movie_dataset["revenue"].notna()).astype(int)
movie_dataset["has_imdb_rating_data"] = movie_dataset["imdbAverageRating"].notna().astype(int)
movie_dataset["has_bechdel_data"] = movie_dataset["bechdelRating"].notna().astype(int)
movie_dataset["has_cast_representation_data"] = movie_dataset["female_ratio_all_cast"].notna().astype(int)
movie_dataset["has_crew_representation_data"] = movie_dataset["female_crew_ratio"].notna().astype(int)
movie_dataset["has_core_eda_data"] = (
    movie_dataset[["title", "year", "genre1", "female_ratio_all_cast", "female_director"]].notna().all(axis=1)
).astype(int)

financial_mask = movie_dataset["has_financial_data"].eq(1)
movie_dataset["profit"] = np.where(financial_mask, movie_dataset["revenue"] - movie_dataset["budget"], np.nan)
movie_dataset["roi"] = np.where(financial_mask, movie_dataset["revenue"] / movie_dataset["budget"], np.nan)
movie_dataset["log_revenue"] = np.where(movie_dataset["revenue"].notna(), np.log1p(movie_dataset["revenue"]), np.nan)
movie_dataset["log_budget"] = np.where(movie_dataset["budget"].notna(), np.log1p(movie_dataset["budget"]), np.nan)

# Keep movie records with valid title and year fields.
movie_dataset = movie_dataset.dropna(subset=["title", "year"]).copy()
movie_dataset["year"] = movie_dataset["year"].astype(int)

print("Final processed dataset:", movie_dataset.shape)
movie_dataset.head()

Final processed dataset: (45346, 61)


,tmdb_id,imdb_id,title,title_clean,year,release_decade,release_date,budget,revenue,popularity,...,has_financial_data,has_imdb_rating_data,has_bechdel_data,has_cast_representation_data,has_crew_representation_data,has_core_eda_data,profit,roi,log_revenue,log_budget
0,862,tt0114709,Toy Story,toy story,1995,1990,1995-10-30,30000000.0,373554033.0,21.946943,...,1,1,1,1,1,1,343554033.0,12.451801,19.738573,17.216708
1,8844,tt0113497,Jumanji,jumanji,1995,1990,1995-12-15,65000000.0,262797249.0,17.015539,...,1,1,1,1,1,1,197797249.0,4.043035,19.386893,17.989898
2,15602,tt0113228,Grumpier Old Men,grumpier old men,1995,1990,1995-12-22,NaN,NaN,11.712900,...,0,1,1,1,1,1,NaN,NaN,NaN,NaN
3,31357,tt0114885,Waiting to Exhale,waiting to exhale,1995,1990,1995-12-22,16000000.0,81452156.0,3.859495,...,1,1,1,1,1,1,65452156.0,5.090760,18.215526,16.588099
4,11862,tt0113041,Father of the Bride Part II,father of the bride part ii,1995,1990,1995-02-10,NaN,76578911.0,8.387519,...,0,0,0,1,1,1,NaN,NaN,18.153832,NaN


In [10]:
# Save the full enriched dataset and the cleaned modeling dataset
full_path = OUT_DIR / "movie_dataset_enriched.csv"
compact_path = OUT_DIR / "cleaned_movie_dataset.csv"

# Select columns for the compact cleaned dataset.
analysis_cols = [
    "tmdb_id", "imdb_id", "movieId", "title", "year", "release_decade",
    "budget_raw", "revenue_raw", "budget", "revenue", "profit", "roi", "log_budget", "log_revenue",
    "popularity", "runtime", "vote_average", "vote_count", "movielens_avg_rating", "movielens_rating_count",
    "imdbAverageRating", "numVotes", "bechdelRating", "bechdel_pass",
    "genre1", "genre2", "genre3", "original_language", "production_country", "main_production_company",
    "female_ratio_all_cast", "female_ratio_top3", "female_ratio_top5", "female_ratio_top10",
    "female_lead", "female_second_billed", "female_director", "female_writer", "female_producer",
    "female_executive_producer", "female_editor", "female_cinematographer", "female_crew_ratio",
    "female_creative_leadership_score",
    "oscar_nomination_count", "oscar_win_count", "oscar_directing_nomination", "oscar_directing_win",
    "oscar_best_picture_nomination", "oscar_best_picture_win",
    "has_budget_data", "has_revenue_data", "has_financial_data", "has_imdb_rating_data", "has_bechdel_data",
    "has_cast_representation_data", "has_crew_representation_data", "has_core_eda_data",
]
# Select columns for the compact cleaned dataset.
analysis_cols = [col for col in analysis_cols if col in movie_dataset.columns]

# Write processed datasets to disk.
movie_dataset.to_csv(full_path, index=False)
movie_dataset[analysis_cols].to_csv(compact_path, index=False)

# Build a processing summary with core coverage counts.
summary = pd.DataFrame({
    "metric": [
        "rows", "movies_with_credits", "movies_with_movielens_ratings", "movies_with_bechdel",
        "movies_with_oscar_nomination", "movies_with_positive_revenue", "movies_with_positive_budget",
        "movies_with_complete_financial_data", "movies_with_core_eda_data", "movies_with_imdb_rating_data",
    ],
    "value": [
        len(movie_dataset),
        int(movie_dataset["female_ratio_all_cast"].notna().sum()),
        int(movie_dataset["movielens_avg_rating"].notna().sum()),
        int(movie_dataset["bechdelRating"].notna().sum()),
        int((movie_dataset["oscar_nomination_count"] > 0).sum()),
        int(movie_dataset["has_revenue_data"].sum()),
        int(movie_dataset["has_budget_data"].sum()),
        int(movie_dataset["has_financial_data"].sum()),
        int(movie_dataset["has_core_eda_data"].sum()),
        int(movie_dataset["has_imdb_rating_data"].sum()),
    ],
})
summary.to_csv(OUT_DIR / "processing_summary.csv", index=False)

# Print output paths and display the processing summary.
print("Saved:")
print(full_path)
print(compact_path)
print(OUT_DIR / "processing_summary.csv")
summary

Saved:
..\data\processed\movie_dataset_enriched.csv
..\data\processed\cleaned_movie_dataset.csv
..\data\processed\processing_summary.csv


,metric,value
0,rows,45346
1,movies_with_credits,40318
2,movies_with_movielens_ratings,44626
3,movies_with_bechdel,7709
4,movies_with_oscar_nomination,2867
5,movies_with_positive_revenue,7397
6,movies_with_positive_budget,8876
7,movies_with_complete_financial_data,5375
8,movies_with_core_eda_data,38855
9,movies_with_imdb_rating_data,7709
